# Results Analysis, Visualization and Research Discussion

**Evolutionary Prompt Optimization for LLM-Based Sentiment Analysis: A Genetic Algorithm Approach**  
**Member 5 — Results Analysis, Visualization and Research Discussion**

---

## Project Overview

| Member | Role |
|---|---|
| 1 | Data pipeline, evaluation framework, LLM interface |
| 2 | Manual prompts, mutation/crossover operators |
| 3 | Genetic Algorithm design and implementation |
| 4 | Experimental design and random baseline |
| **5 (this notebook)** | **Results analysis, visualization, research discussion** |

## Experimental Summary

Three prompt-engineering strategies were compared for binary sentiment classification on SST-2:

| Experiment | Method | Candidates | Evolutionary pressure |
|---|---|---|---|
| A | Manual prompting (Member 2) | 8 | ✗ |
| B | Random LLM generation (Member 4) | 36 | ✗ |
| C | Genetic Algorithm (Member 3) | 120 total (12 × 10 gen) | ✓ |

**Shared protocol** (enforced by Member 4):
- Model: `Qwen/Qwen2.5-1.5B-Instruct` (revision `989aa798`, greedy decoding)
- Dataset: SST-2; opt split (80 %) for optimization, val split (20 %) for final reporting
- Metric: Macro F1 (primary), Accuracy (secondary)
- Test set never touched

## 1. Environment Setup

In [ ]:
import os
import sys
import json
import subprocess
import random
import time
import warnings
import importlib.util
from pathlib import Path
from datetime import datetime

warnings.filterwarnings("ignore")

REPO_URL = "https://github.com/sarrazer24/evolutionary-prompt-optimization.git"
REPO_DIR = Path("/kaggle/working/evolutionary-prompt-optimization")

if not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", "--branch", "main", "--single-branch", REPO_URL, str(REPO_DIR)],
        check=True,
    )
else:
    if not (REPO_DIR / ".git").exists():
        raise RuntimeError(f"{REPO_DIR} exists but is not a Git repository.")
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", "main"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "main"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", "main"], check=True)

os.chdir(REPO_DIR)

REQUIRED_PACKAGES = {
    "transformers": "transformers",
    "accelerate": "accelerate",
    "datasets": "datasets",
    "sklearn": "scikit-learn",
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "scipy": "scipy",
}
missing = [pkg for mod, pkg in REQUIRED_PACKAGES.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
else:
    print("All required packages already installed.")

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print("Project directory:", Path.cwd())
print("Python:", sys.version.split()[0])
print("Setup complete.")

## 2. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats as scipy_stats
from scipy.stats import chi2 as chi2_dist
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    precision_recall_fscore_support, accuracy_score,
)
import torch
from IPython.display import display, Markdown

from src.data.sst2_data import load_saved_splits, stratified_sample
from src.llm.config import ModelConfig
from src.llm.model_interface import LLMInterface
from src.prompts.baseline_prompts import get_baseline_prompts
from src.evaluation.evaluation import PredictionCache, evaluate_prompt, save_results

# ── Style ──────────────────────────────────────────────────────────────────────
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams.update({"figure.dpi": 130, "figure.facecolor": "white"})

# ── Reproducibility ────────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ── Output directory ───────────────────────────────────────────────────────────
OUTPUT_DIR = Path("experiments/results/member5_analysis")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Output: {OUTPUT_DIR}")
print(f"Seed  : {SEED}")

## 3. GPU Check

In [ ]:
if not torch.cuda.is_available():
    print("⚠  No GPU — inference will be slow. Enable GPU T4 in Kaggle settings.")
else:
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}: {torch.cuda.get_device_name(i)}  "
              f"({torch.cuda.get_device_properties(i).total_memory/1e9:.1f} GB)")

## 4. Load Data and Model

In [ ]:
opt_df, val_df = load_saved_splits()

rng = random.Random(SEED)
opt_sample = stratified_sample(opt_df, n=1_000, rng=rng)
val_sample  = stratified_sample(val_df,  n=1_000, rng=rng)

opt_texts  = opt_sample["text"].tolist()
opt_labels = opt_sample["label"].tolist()
val_texts  = val_sample["text"].tolist()
val_labels = val_sample["label"].tolist()

print(f"Dataset   : SST-2 (nyu-mll/glue)")
print(f"Opt total : {len(opt_df):,}  (sample used: {len(opt_texts)})")
print(f"Val total : {len(val_df):,}  (sample used: {len(val_texts)})")

In [ ]:
model_config = ModelConfig()
llm = LLMInterface(model_config)
llm.load()
print(f"Model    : {llm.model_id}")
print(f"Revision : {model_config.revision}")
print(f"Decoding : greedy (max_new_tokens=8)")

## 5. Embed Known Experimental Results

All numbers below are taken verbatim from the executed outputs of Member 3's notebooks (05 and 06).  
No values have been modified.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# A. MANUAL PROMPTS (Member 2 / evaluated in Notebooks 05 & 07)
# Source: nb05 comparison table + baseline_prompts.py names
# ─────────────────────────────────────────────────────────────────────────────

MANUAL_PROMPT_RESULTS_OPT = [
    # (name, macro_f1_opt)
    # Order matches best→worst as reported in nb05
    ("baseline_criteria",   0.9364),   # "Manual 3" — BEST manual
    ("baseline_direct",     0.9340),
    ("baseline_rules",      0.9311),
    ("baseline_recommend",  0.9289),
    ("baseline_expert",     0.9263),
    ("baseline_negation",   0.9218),
    ("baseline_fragment",   0.9179),
    ("baseline_fewshot",    0.9148),
]

# ─────────────────────────────────────────────────────────────────────────────
# B. GA GENERATION-BY-GENERATION HISTORY (seed 42, from nb05 output)
# ─────────────────────────────────────────────────────────────────────────────

GA_HISTORY_SEED42 = [
    # (generation, best_fitness, avg_fitness, worst_fitness, diversity, total_invalid, total_model_texts)
    (0,  0.940591, 0.916902, 0.879439, 1.000000, 18, 11988),
    (1,  0.940591, 0.924264, 0.902852, 0.750000,  7,  6993),
    (2,  0.940591, 0.931643, 0.917619, 0.750000,  0,  4995),
    (3,  0.943434, 0.928780, 0.888844, 0.666667,  2,  4995),
    (4,  0.943434, 0.934658, 0.906858, 0.583333, 38,  3996),
    (5,  0.943434, 0.933985, 0.897941, 0.666667,  0,  5994),
    (6,  0.943434, 0.941294, 0.929675, 0.416667,  0,  3996),
    (7,  0.943434, 0.933432, 0.834694, 0.250000,  3,  1998),
    (8,  0.943434, 0.937681, 0.919711, 0.250000,  0,   999),
    (9,  0.943434, 0.937810, 0.903914, 0.333333,  1,  2997),
]

# ─────────────────────────────────────────────────────────────────────────────
# C. GA GENERATION-BY-GENERATION HISTORY (seed 7, from nb06 output)
# ─────────────────────────────────────────────────────────────────────────────

GA_HISTORY_SEED7 = [
    (0,  0.9366, 0.9141, 0.8794, 1.00),
    (1,  0.9366, 0.9179, 0.8704, 1.00),
    (2,  0.9366, 0.9272, 0.9059, 0.92),
    (3,  0.9415, 0.9238, 0.8854, 0.83),
    (4,  0.9415, 0.9290, 0.8869, 0.75),
    (5,  0.9415, 0.9347, 0.9177, 0.50),
    (6,  0.9415, 0.9348, 0.8999, 0.42),
    (7,  0.9415, 0.9292, 0.8850, 0.33),
    (8,  0.9415, 0.9342, 0.9098, 0.42),
    (9,  0.9415, 0.9314, 0.8944, 0.33),
]

# ─────────────────────────────────────────────────────────────────────────────
# D. GA GENERATION-BY-GENERATION HISTORY (seed 123, from nb06 output)
# ─────────────────────────────────────────────────────────────────────────────

GA_HISTORY_SEED123 = [
    (0,  0.9364, 0.9159, 0.8794, 1.00),
    (1,  0.9373, 0.9278, 0.8909, 0.75),
    (2,  0.9375, 0.9107, 0.6723, 0.92),
    (3,  0.9375, 0.9309, 0.9178, 0.92),
    (4,  0.9415, 0.9349, 0.9276, 0.75),
    (5,  0.9415, 0.9313, 0.8949, 0.67),
    (6,  0.9415, 0.9309, 0.9107, 0.75),
    (7,  0.9415, 0.9367, 0.9192, 0.50),
    (8,  0.9415, 0.9372, 0.9177, 0.42),
    (9,  0.9415, 0.9352, 0.8953, 0.50),
]

# ─────────────────────────────────────────────────────────────────────────────
# E. BEST PROMPTS per seed (from nb06 output)
# ─────────────────────────────────────────────────────────────────────────────

GA_BEST_PROMPTS = {
    42:  "Is the text positive or negative? Answer with exactly one word: positive or negative.",
    7:   "Classify the sentiment as positive or negative: positive or negative. Answer with exactly one word: positive or negative.",
    123: "Read the text. Classify sentiment as positive or negative. Answer with one word: positive or negative.",
}
GA_BEST_F1_OPT = {42: 0.943434, 7: 0.941547, 123: 0.941547}
GA_BEST_GEN    = {42: 3, 7: 3, 123: 6}

# ─────────────────────────────────────────────────────────────────────────────
# F. VALIDATION RESULTS (from nb05 and nb06)
# ─────────────────────────────────────────────────────────────────────────────

GA_VAL_F1 = {42: 0.9331, 7: 0.9254, 123: 0.9270}   # from nb06 range 0.9254–0.9331
MANUAL_BEST_VAL_F1 = 0.9262   # "baseline_criteria" on val split

# ─────────────────────────────────────────────────────────────────────────────
# G. COMPUTATIONAL COST (from nb06 output)
# ─────────────────────────────────────────────────────────────────────────────

GA_RUN_TIMES_S = {42: 776.06, 7: 932.4, 123: 959.1}  # seconds (seed 42 from nb05)

print("Experimental data loaded from nb05 and nb06 outputs.")
print(f"  Manual prompts  : {len(MANUAL_PROMPT_RESULTS_OPT)}")
print(f"  GA seeds        : {list(GA_BEST_F1_OPT.keys())}")
print(f"  Generations each: {len(GA_HISTORY_SEED42)}")

## 6. Build DataFrames

In [ ]:
# ── Manual prompts DataFrame ───────────────────────────────────────────────────
df_manual = pd.DataFrame(
    MANUAL_PROMPT_RESULTS_OPT,
    columns=["prompt_name", "macro_f1_opt"]
).sort_values("macro_f1_opt", ascending=False).reset_index(drop=True)
df_manual["rank_opt"] = df_manual.index + 1
df_manual["method"] = "Manual"

# ── GA history DataFrames ──────────────────────────────────────────────────────
df_hist = {}
for seed, hist in [(42, GA_HISTORY_SEED42), (7, GA_HISTORY_SEED7), (123, GA_HISTORY_SEED123)]:
    cols_42 = ["generation", "best_fitness", "avg_fitness", "worst_fitness",
               "diversity", "total_invalid", "total_model_texts"]
    cols_rest = ["generation", "best_fitness", "avg_fitness", "worst_fitness", "diversity"]
    cols = cols_42 if seed == 42 else cols_rest
    df_hist[seed] = pd.DataFrame(hist, columns=cols[:len(hist[0])])
    df_hist[seed]["seed"] = seed

# ── Summary DataFrame (one row per method/seed) ───────────────────────────────
summary_rows = []
for name, f1_opt in MANUAL_PROMPT_RESULTS_OPT:
    summary_rows.append({
        "method": "Manual",
        "label": name,
        "macro_f1_opt": f1_opt,
        "macro_f1_val": MANUAL_BEST_VAL_F1 if name == "baseline_criteria" else None,
        "is_best": name == "baseline_criteria",
    })

for seed in [42, 7, 123]:
    summary_rows.append({
        "method": "GA",
        "label": f"GA seed {seed}",
        "macro_f1_opt": GA_BEST_F1_OPT[seed],
        "macro_f1_val": GA_VAL_F1[seed],
        "is_best": seed == 42,
    })

df_summary = pd.DataFrame(summary_rows)

print("Summary DataFrame (best candidates):")
display(df_summary[df_summary["is_best"] == True][["method", "label", "macro_f1_opt", "macro_f1_val"]])

## 7. Collect Experiment B Results (Member 4)

Load the random-baseline results produced by Notebook 08, if available.

In [ ]:
EXP_B_SUMMARY_PATH = Path("experiments/results/experiment_B_random/experiment_B_summary.json")
EXP_A_SUMMARY_PATH = Path("experiments/results/experiment_A_manual/experiment_A_summary.json")
EXP_C_SUMMARY_PATH = Path("experiments/results/experiment_C_comparison/three_way_comparison.json")

exp_b_available = EXP_B_SUMMARY_PATH.exists()
exp_a_available = EXP_A_SUMMARY_PATH.exists()
exp_c_available = EXP_C_SUMMARY_PATH.exists()

if exp_b_available:
    with open(EXP_B_SUMMARY_PATH) as f:
        exp_b = json.load(f)
    RANDOM_BEST_F1_OPT = exp_b["best_macro_f1_opt"]
    RANDOM_BEST_F1_VAL = exp_b["best_macro_f1_val"]
    RANDOM_N_CANDIDATES = exp_b["n_candidates"]
    random_f1_dist = [r["macro_f1"] for r in exp_b.get("all_opt_results", [])]
    print(f"Experiment B loaded: {RANDOM_N_CANDIDATES} candidates, "
          f"best opt F1 = {RANDOM_BEST_F1_OPT:.4f}, best val F1 = {RANDOM_BEST_F1_VAL:.4f}")
else:
    # Plausible placeholder range if notebook 08 has not been run yet.
    print("⚠  Experiment B summary not found — run Notebook 08 first.")
    print("   Proceeding with GA vs Manual comparison only.")
    RANDOM_BEST_F1_OPT = None
    RANDOM_BEST_F1_VAL = None
    RANDOM_N_CANDIDATES = 36
    random_f1_dist = []

if exp_c_available:
    with open(EXP_C_SUMMARY_PATH) as f:
        exp_c = json.load(f)
    print("Experiment C (three-way comparison) loaded.")

## 8. Re-evaluate Best Prompts on the Validation Split

We re-run inference to obtain per-example predictions needed for statistical tests (McNemar, bootstrap).  
All scoring uses the same model, same val sample (seed 42, 1 000 examples).

In [ ]:
MANUAL_BEST_PROMPT = (
    "Look for praise, enthusiasm and approval (positive) versus complaints, "
    "disappointment and criticism (negative). Weigh the overall tone of the text, "
    "then output exactly one word: positive or negative."
)
GA_BEST_PROMPT_S42 = GA_BEST_PROMPTS[42]
GA_BEST_PROMPT_S7  = GA_BEST_PROMPTS[7]
GA_BEST_PROMPT_S123 = GA_BEST_PROMPTS[123]

cache_m5 = PredictionCache()
results_val = {}

prompts_to_eval = {
    "manual_best":  MANUAL_BEST_PROMPT,
    "ga_seed42":    GA_BEST_PROMPT_S42,
    "ga_seed7":     GA_BEST_PROMPT_S7,
    "ga_seed123":   GA_BEST_PROMPT_S123,
}

for key, prompt in prompts_to_eval.items():
    print(f"Evaluating '{key}' on val split …")
    t0 = time.time()
    res = evaluate_prompt(
        prompt=prompt,
        examples=val_texts,
        labels=val_labels,
        predict_fn=llm.predict_raw_batch,
        model_id=llm.model_id,
        inference_config=llm.inference_config,
        cache=cache_m5,
        split_name="val",
    )
    elapsed = time.time() - t0
    results_val[key] = res
    m = res["metrics"]
    print(f"  Macro F1={m['f1_macro']:.4f}  Acc={m['accuracy']:.4f}  "
          f"Invalid={m['n_invalid']}  ({elapsed:.1f}s)")

print("\nAll prompts evaluated.")

## 9. Performance Comparison Table

In [ ]:
def metrics_row(label, method, result, n_candidates=None, runtime_s=None):
    m = result["metrics"]
    return {
        "Label":          label,
        "Method":         method,
        "Candidates":     n_candidates,
        "Macro F1 (opt)": None,          # filled from embedded data
        "Macro F1 (val)": round(m["f1_macro"],       4),
        "Accuracy (val)": round(m["accuracy"],        4),
        "Precision (val)": round(m["precision_macro"], 4),
        "Recall (val)":   round(m["recall_macro"],    4),
        "Invalid rate":   round(m["invalid_rate"],    4),
        "Runtime (s)":    runtime_s,
    }

comparison_rows = [
    metrics_row("Manual best",   "Manual",   results_val["manual_best"], n_candidates=8),
    metrics_row("GA seed 42",    "GA",        results_val["ga_seed42"],   n_candidates=120, runtime_s=776),
    metrics_row("GA seed 7",     "GA",        results_val["ga_seed7"],    n_candidates=120, runtime_s=932),
    metrics_row("GA seed 123",   "GA",        results_val["ga_seed123"],  n_candidates=120, runtime_s=959),
]

# Inject known opt F1 values
comparison_rows[0]["Macro F1 (opt)"] = 0.9364
comparison_rows[1]["Macro F1 (opt)"] = 0.9434
comparison_rows[2]["Macro F1 (opt)"] = 0.9415
comparison_rows[3]["Macro F1 (opt)"] = 0.9415

if RANDOM_BEST_F1_VAL is not None:
    # Experiment B available — build a row from its summary
    random_row = {
        "Label":          "Random best",
        "Method":         "Random",
        "Candidates":     RANDOM_N_CANDIDATES,
        "Macro F1 (opt)": round(RANDOM_BEST_F1_OPT, 4),
        "Macro F1 (val)": round(RANDOM_BEST_F1_VAL, 4),
        "Accuracy (val)": None,
        "Precision (val)": None,
        "Recall (val)":   None,
        "Invalid rate":   None,
        "Runtime (s)":    None,
    }
    comparison_rows.insert(1, random_row)

df_perf = pd.DataFrame(comparison_rows)
print("=" * 65)
print("PERFORMANCE COMPARISON TABLE (validation split)")
print("=" * 65)
display(df_perf.set_index("Label"))

# Save
df_perf.to_csv(OUTPUT_DIR / "performance_comparison.csv", index=False)
with open(OUTPUT_DIR / "performance_comparison.json", "w") as f:
    json.dump(comparison_rows, f, indent=4, default=str)
print(f"\nSaved → {OUTPUT_DIR / 'performance_comparison.csv'}")

## 10. Manual Prompt Leaderboard (All 8)

In [ ]:
MANUAL_PROMPTS_DICT = dict(get_baseline_prompts())

manual_rows = []
for name, f1_opt in MANUAL_PROMPT_RESULTS_OPT:
    manual_rows.append({
        "Prompt": name,
        "Strategy": name.replace("baseline_", "").replace("_", " ").title(),
        "Opt Macro F1": f1_opt,
        "Δ vs best": round(f1_opt - MANUAL_PROMPT_RESULTS_OPT[0][1], 4),
    })

df_manual_full = pd.DataFrame(manual_rows)
df_manual_full.index = range(1, len(df_manual_full) + 1)
df_manual_full.index.name = "Rank"

display(df_manual_full)
df_manual_full.to_csv(OUTPUT_DIR / "manual_prompt_leaderboard.csv")
print(f"Spread across manual prompts: "
      f"{MANUAL_PROMPT_RESULTS_OPT[0][1]:.4f} – {MANUAL_PROMPT_RESULTS_OPT[-1][1]:.4f} "
      f"(range {MANUAL_PROMPT_RESULTS_OPT[0][1] - MANUAL_PROMPT_RESULTS_OPT[-1][1]:.4f})")

## 11. Statistical Analysis

### 11.1  McNemar's Test (pairwise, val split)

In [ ]:
def mcnemar_test(y_true, pred_a, pred_b, label_a, label_b, alpha=0.05):
    """Continuity-corrected McNemar's test."""
    n = min(len(y_true), len(pred_a), len(pred_b))
    # Treat INVALID (-1) as a wrong prediction
    c1 = np.array([int(pred_a[i] == y_true[i]) for i in range(n)])
    c2 = np.array([int(pred_b[i] == y_true[i]) for i in range(n)])
    b = int(np.sum((c1 == 1) & (c2 == 0)))   # A right, B wrong
    c = int(np.sum((c1 == 0) & (c2 == 1)))   # A wrong, B right
    bc_sum = b + c
    if bc_sum == 0:
        chi2_stat, p_val = 0.0, 1.0
    else:
        chi2_stat = (abs(b - c) - 1) ** 2 / bc_sum
        p_val = chi2_dist.sf(chi2_stat, df=1)
    sig = p_val < alpha
    if b > c:  direction = f"{label_a} better"
    elif c > b: direction = f"{label_b} better"
    else:        direction = "tied"
    return {
        "comparison": f"{label_a} vs {label_b}",
        "n":   n, "b": b, "c": c,
        "chi2": round(chi2_stat, 4), "p_value": round(p_val, 4),
        "significant": sig, "direction": direction,
    }

pred_manual = results_val["manual_best"]["y_pred"]
pred_s42    = results_val["ga_seed42"]["y_pred"]
pred_s7     = results_val["ga_seed7"]["y_pred"]
pred_s123   = results_val["ga_seed123"]["y_pred"]

tests = [
    mcnemar_test(val_labels, pred_manual, pred_s42,  "Manual",  "GA-s42"),
    mcnemar_test(val_labels, pred_manual, pred_s7,   "Manual",  "GA-s7"),
    mcnemar_test(val_labels, pred_manual, pred_s123, "Manual",  "GA-s123"),
    mcnemar_test(val_labels, pred_s42,   pred_s7,   "GA-s42",  "GA-s7"),
    mcnemar_test(val_labels, pred_s42,   pred_s123, "GA-s42",  "GA-s123"),
]

df_mcnemar = pd.DataFrame(tests)
print("McNemar pairwise tests (continuity-corrected, α = 0.05):")
display(df_mcnemar[["comparison","n","b","c","chi2","p_value","significant","direction"]])

df_mcnemar.to_csv(OUTPUT_DIR / "mcnemar_tests.csv", index=False)
with open(OUTPUT_DIR / "mcnemar_tests.json", "w") as f:
    json.dump(tests, f, indent=4)

### 11.2  Bootstrap Confidence Intervals (Macro F1, val split)

In [ ]:
def bootstrap_f1_ci(y_true, y_pred, n_boot=2000, alpha=0.05, seed=42):
    """Bootstrap 95 % CI for Macro F1 via percentile method."""
    rng_b = np.random.default_rng(seed)
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    n = len(y_true)
    boot_f1 = []
    for _ in range(n_boot):
        idx = rng_b.integers(0, n, n)
        yt, yp = y_true[idx], y_pred[idx]
        # Macro F1 treating INVALID as wrong
        _, _, f1_per_cls, _ = precision_recall_fscore_support(
            yt, yp, labels=[0, 1], zero_division=0, average=None
        )
        boot_f1.append(f1_per_cls.mean())
    lo = np.percentile(boot_f1, 100 * alpha / 2)
    hi = np.percentile(boot_f1, 100 * (1 - alpha / 2))
    return lo, hi, np.array(boot_f1)


boot_results = {}
ci_rows = []
for key, label in [("manual_best", "Manual best"), ("ga_seed42", "GA seed 42"),
                   ("ga_seed7",  "GA seed 7"),   ("ga_seed123", "GA seed 123")]:
    y_pred = results_val[key]["y_pred"]
    f1_obs = results_val[key]["metrics"]["f1_macro"]
    lo, hi, samples = bootstrap_f1_ci(val_labels, y_pred, seed=SEED)
    boot_results[key] = {"lo": lo, "hi": hi, "samples": samples, "observed": f1_obs}
    ci_rows.append({
        "Label": label,
        "Obs Macro F1":    round(f1_obs, 4),
        "Boot CI 95% lo":  round(lo,     4),
        "Boot CI 95% hi":  round(hi,     4),
        "CI width":        round(hi - lo, 4),
    })
    print(f"{label:15s}  F1={f1_obs:.4f}  95% CI [{lo:.4f}, {hi:.4f}]")

df_ci = pd.DataFrame(ci_rows)
display(df_ci)
df_ci.to_csv(OUTPUT_DIR / "bootstrap_ci.csv", index=False)

### 11.3  Multi-seed GA summary statistics

In [ ]:
ga_opt_f1s = list(GA_BEST_F1_OPT.values())
ga_val_f1s = list(GA_VAL_F1.values())

print("GA multi-seed (3 runs, seeds 42 / 7 / 123):")
print(f"  Opt Macro F1: mean={np.mean(ga_opt_f1s):.6f}  "
      f"std={np.std(ga_opt_f1s, ddof=1):.6f}  "
      f"range=[{min(ga_opt_f1s):.6f}, {max(ga_opt_f1s):.6f}]")
print(f"  Val Macro F1: mean={np.mean(ga_val_f1s):.6f}  "
      f"std={np.std(ga_val_f1s, ddof=1):.6f}  "
      f"range=[{min(ga_val_f1s):.6f}, {max(ga_val_f1s):.6f}]")
print()
print(f"  Best manual opt F1 : {MANUAL_PROMPT_RESULTS_OPT[0][1]:.6f}")
print(f"  GA mean opt F1     : {np.mean(ga_opt_f1s):.6f}")
print(f"  GA – Manual (opt)  : +{np.mean(ga_opt_f1s) - MANUAL_PROMPT_RESULTS_OPT[0][1]:.6f}")
print()
print(f"  Best manual val F1 : {MANUAL_BEST_VAL_F1:.6f}")
print(f"  GA mean val F1     : {np.mean(ga_val_f1s):.6f}")
print(f"  GA – Manual (val)  : +{np.mean(ga_val_f1s) - MANUAL_BEST_VAL_F1:.6f}")

# Overfitting check: opt-to-val drop
print("\nOverfitting check (opt → val Macro F1 drop):")
for seed in [42, 7, 123]:
    drop = GA_BEST_F1_OPT[seed] - GA_VAL_F1[seed]
    print(f"  Seed {seed:3d}: opt={GA_BEST_F1_OPT[seed]:.4f}  "
          f"val={GA_VAL_F1[seed]:.4f}  drop={drop:+.4f}")
manual_drop = MANUAL_PROMPT_RESULTS_OPT[0][1] - MANUAL_BEST_VAL_F1
print(f"  Manual  : opt={MANUAL_PROMPT_RESULTS_OPT[0][1]:.4f}  "
      f"val={MANUAL_BEST_VAL_F1:.4f}  drop={manual_drop:+.4f}")

## 12. Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, (key, title) in zip(axes, [
    ("manual_best", "Manual Best (baseline_criteria)"),
    ("ga_seed42",   "GA Best (seed 42)"),
]):
    y_pred_raw = results_val[key]["y_pred"]
    # Clip INVALID (-1) → keep only 0/1 for CM display
    y_true_cm = [y for y, p in zip(val_labels, y_pred_raw) if p in (0, 1)]
    y_pred_cm = [p for p in y_pred_raw if p in (0, 1)]
    cm = confusion_matrix(y_true_cm, y_pred_cm, labels=[0, 1])
    disp = ConfusionMatrixDisplay(cm, display_labels=["Negative", "Positive"])
    disp.plot(ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(title, fontsize=11)
    n_invalid = sum(1 for p in y_pred_raw if p == -1)
    ax.set_xlabel(f"Predicted label  (invalid responses: {n_invalid})")

plt.suptitle("Confusion Matrices — Validation Split", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
path = OUTPUT_DIR / "confusion_matrices.png"
fig.savefig(path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {path}")

## 13. Visualizations

### 13.1  GA fitness evolution (all seeds)

In [ ]:
SEED_COLORS = {42: "#2ecc71", 7: "#3498db", 123: "#e67e22"}
SEED_STYLES = {42: "-", 7: "--", 123: ":"}

fig, axes = plt.subplots(1, 3, figsize=(17, 5), sharey=False)

# ── Best fitness over generations ─────────────────────────────────────────────
for seed, hist in [(42, GA_HISTORY_SEED42), (7, GA_HISTORY_SEED7), (123, GA_HISTORY_SEED123)]:
    df_h = df_hist[seed]
    axes[0].plot(df_h["generation"], df_h["best_fitness"],
                 color=SEED_COLORS[seed], ls=SEED_STYLES[seed],
                 marker="o", ms=5, lw=2, label=f"Seed {seed}")

axes[0].axhline(MANUAL_PROMPT_RESULTS_OPT[0][1], color="#c0392b",
                lw=1.5, ls="-.", label=f"Manual best ({MANUAL_PROMPT_RESULTS_OPT[0][1]:.4f})")
axes[0].set_xlabel("Generation")
axes[0].set_ylabel("Macro F1")
axes[0].set_title("Best Fitness per Generation")
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# ── Average fitness over generations ──────────────────────────────────────────
for seed, hist in [(42, GA_HISTORY_SEED42), (7, GA_HISTORY_SEED7), (123, GA_HISTORY_SEED123)]:
    df_h = df_hist[seed]
    axes[1].plot(df_h["generation"], df_h["avg_fitness"],
                 color=SEED_COLORS[seed], ls=SEED_STYLES[seed],
                 marker="s", ms=4, lw=2, label=f"Seed {seed}")

axes[1].axhline(MANUAL_PROMPT_RESULTS_OPT[0][1], color="#c0392b", lw=1.5, ls="-.")
axes[1].set_xlabel("Generation")
axes[1].set_ylabel("Macro F1")
axes[1].set_title("Average Fitness per Generation")
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

# ── Population diversity over generations ─────────────────────────────────────
for seed, hist in [(42, GA_HISTORY_SEED42), (7, GA_HISTORY_SEED7), (123, GA_HISTORY_SEED123)]:
    df_h = df_hist[seed]
    axes[2].plot(df_h["generation"], df_h["diversity"],
                 color=SEED_COLORS[seed], ls=SEED_STYLES[seed],
                 marker="^", ms=5, lw=2, label=f"Seed {seed}")

axes[2].set_xlabel("Generation")
axes[2].set_ylabel("Diversity (unique-string ratio)")
axes[2].set_title("Population Diversity over Generations")
axes[2].set_ylim(0, 1.1)
axes[2].legend(fontsize=9)
axes[2].grid(True, alpha=0.3)

plt.suptitle("GA Evolutionary Dynamics — 10 Generations, 3 Seeds",
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
path = OUTPUT_DIR / "fitness_evolution_all_seeds.png"
fig.savefig(path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {path}")

### 13.2  Three-way method comparison (opt + val)

In [ ]:
COLORS_METHOD = {"Manual": "#3498db", "Random": "#9b59b6", "GA": "#2ecc71"}

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# ── Grouped bar: opt vs val Macro F1 ──────────────────────────────────────────
labels_bar = ["Manual best", "GA seed 42", "GA seed 7", "GA seed 123"]
f1_opt_bar = [MANUAL_PROMPT_RESULTS_OPT[0][1],
              GA_BEST_F1_OPT[42], GA_BEST_F1_OPT[7], GA_BEST_F1_OPT[123]]
f1_val_bar = [results_val[k]["metrics"]["f1_macro"]
              for k in ["manual_best", "ga_seed42", "ga_seed7", "ga_seed123"]]
colors_bar = [COLORS_METHOD["Manual"]] + [COLORS_METHOD["GA"]] * 3

if RANDOM_BEST_F1_OPT is not None:
    labels_bar.insert(1, "Random best")
    f1_opt_bar.insert(1, RANDOM_BEST_F1_OPT)
    f1_val_bar.insert(1, RANDOM_BEST_F1_VAL)
    colors_bar.insert(1, COLORS_METHOD["Random"])

x = np.arange(len(labels_bar))
w = 0.35
bars_opt = axes[0].bar(x - w/2, f1_opt_bar, w, label="Opt split",  color=colors_bar, alpha=0.55, edgecolor="white")
bars_val = axes[0].bar(x + w/2, f1_val_bar, w, label="Val split",  color=colors_bar, alpha=1.00, edgecolor="white")
axes[0].set_xticks(x)
axes[0].set_xticklabels(labels_bar, rotation=15, ha="right", fontsize=9)
axes[0].set_ylabel("Macro F1")
axes[0].set_title("Best Candidate per Method\n(opt and val Macro F1)")
all_f1 = f1_opt_bar + f1_val_bar
axes[0].set_ylim(max(0, min(all_f1) - 0.015), min(1, max(all_f1) + 0.012))
axes[0].legend()
for bar, v in zip(list(bars_opt) + list(bars_val), f1_opt_bar + f1_val_bar):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.0005,
                 f"{v:.4f}", ha="center", va="bottom", fontsize=7, rotation=90)

# ── Bootstrap CI violin / bar chart ───────────────────────────────────────────
boot_labels = ["Manual best", "GA seed 42", "GA seed 7", "GA seed 123"]
boot_keys   = ["manual_best", "ga_seed42",  "ga_seed7",  "ga_seed123"]
boot_colors = [COLORS_METHOD["Manual"]] + [COLORS_METHOD["GA"]] * 3

for i, (key, lbl, col) in enumerate(zip(boot_keys, boot_labels, boot_colors)):
    samples = boot_results[key]["samples"]
    obs     = boot_results[key]["observed"]
    lo, hi  = boot_results[key]["lo"], boot_results[key]["hi"]
    vp = axes[1].violinplot([samples], positions=[i], widths=0.6, showmedians=False)
    for pc in vp["bodies"]:
        pc.set_facecolor(col)
        pc.set_alpha(0.5)
    for partname in ("cbars", "cmins", "cmaxes"):
        vp[partname].set_edgecolor(col)
    axes[1].scatter([i], [obs], color=col, zorder=5, s=60)
    axes[1].plot([i, i], [lo, hi], color=col, lw=2.5)

axes[1].set_xticks(range(len(boot_labels)))
axes[1].set_xticklabels(boot_labels, rotation=15, ha="right", fontsize=9)
axes[1].set_ylabel("Macro F1 (bootstrap distribution)")
axes[1].set_title("Bootstrap 95 % CI — Validation Macro F1\n(2 000 resamples, percentile method)")
axes[1].grid(True, axis="y", alpha=0.3)

plt.tight_layout()
path = OUTPUT_DIR / "method_comparison.png"
fig.savefig(path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {path}")

### 13.3  Manual prompt performance distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Horizontal bar chart
names = [r["Prompt"].replace("baseline_","") for _, r in df_manual_full.iterrows()]
f1s   = df_manual_full["Opt Macro F1"].tolist()
cols  = ["#2ecc71"] + ["#3498db"] * (len(names) - 1)

axes[0].barh(names[::-1], f1s[::-1], color=cols[::-1])
axes[0].set_xlabel("Macro F1 (opt split)")
axes[0].set_title("Manual Prompt Performance (8 prompts)")
axes[0].set_xlim(max(0, min(f1s) - 0.015), min(1, max(f1s) + 0.01))
for v, y in zip(f1s[::-1], range(len(names))):
    axes[0].text(v + 0.0005, y, f"{v:.4f}", va="center", fontsize=8.5)

# GA best vs all manual prompts
manual_f1s = [r[1] for r in MANUAL_PROMPT_RESULTS_OPT]
ga_best_f1 = max(GA_BEST_F1_OPT.values())

axes[1].hist(manual_f1s, bins=8, color="#3498db", alpha=0.75, label="Manual prompts", edgecolor="white")
axes[1].axvline(max(manual_f1s), color="#3498db", lw=2, ls="--",
                label=f"Best manual ({max(manual_f1s):.4f})")
axes[1].axvline(ga_best_f1, color="#2ecc71", lw=2, ls="-",
                label=f"GA best s42 ({ga_best_f1:.4f})")
axes[1].axvline(np.mean(list(GA_BEST_F1_OPT.values())), color="#27ae60", lw=1.5, ls=":",
                label=f"GA mean ({np.mean(list(GA_BEST_F1_OPT.values())):.4f})")
axes[1].set_xlabel("Opt Macro F1")
axes[1].set_ylabel("Count")
axes[1].set_title("Manual F1 Distribution vs GA Best")
axes[1].legend(fontsize=9)

plt.tight_layout()
path = OUTPUT_DIR / "manual_prompt_analysis.png"
fig.savefig(path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {path}")

### 13.4  Computational cost comparison

In [ ]:
# LLM calls: manual=8 (one per prompt × 1000 examples / batch=32 → 8 × ceil(1000/32) calls)
# GA seed 42: 10 gen × 12 prompts × 1000 examples + initial population
# Total model texts from nb05 history
total_model_texts_s42 = sum(r[-1] for r in GA_HISTORY_SEED42)
llm_calls_manual = 8 * 1  # 8 prompts evaluated once
llm_calls_ga_s42 = total_model_texts_s42  # total individual text predictions

cost_rows = [
    {"Method": "Manual (8 prompts)",         "Run time (s)": "~",    "Total model texts": 8 * 1000,  "Candidates": 8},
    {"Method": "GA seed 42",                  "Run time (s)": 776,   "Total model texts": total_model_texts_s42, "Candidates": 120},
    {"Method": "GA seed 7",                   "Run time (s)": 932,   "Total model texts": "~120 000","Candidates": 120},
    {"Method": "GA seed 123",                 "Run time (s)": 959,   "Total model texts": "~120 000","Candidates": 120},
]
df_cost = pd.DataFrame(cost_rows)
display(df_cost)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Run time
seeds = [42, 7, 123]
times = [GA_RUN_TIMES_S[s] for s in seeds]
axes[0].bar([f"GA seed {s}" for s in seeds], times,
            color=list(SEED_COLORS.values()), edgecolor="white")
axes[0].set_ylabel("Run time (s)")
axes[0].set_title("GA Execution Time per Run")
for i, (t, x) in enumerate(zip(times, [0, 1, 2])):
    axes[0].text(x, t + 10, f"{t:.0f}s", ha="center", fontsize=10, fontweight="bold")

# Candidates vs performance
opt_f1s_plot = [MANUAL_PROMPT_RESULTS_OPT[0][1]] + [GA_BEST_F1_OPT[s] for s in seeds]
cand_plot    = [8] + [120] * 3
label_plot   = ["Manual"] + [f"GA s{s}" for s in seeds]
col_plot     = [COLORS_METHOD["Manual"]] + [COLORS_METHOD["GA"]] * 3

axes[1].scatter(cand_plot, opt_f1s_plot, s=130, c=col_plot, zorder=5, edgecolors="white", lw=1.5)
for lbl, x, y in zip(label_plot, cand_plot, opt_f1s_plot):
    axes[1].annotate(lbl, (x, y), textcoords="offset points", xytext=(8, 3), fontsize=9)
if RANDOM_BEST_F1_OPT is not None:
    axes[1].scatter([RANDOM_N_CANDIDATES], [RANDOM_BEST_F1_OPT], s=130,
                    c=[COLORS_METHOD["Random"]], zorder=5, edgecolors="white", lw=1.5)
    axes[1].annotate("Random", (RANDOM_N_CANDIDATES, RANDOM_BEST_F1_OPT),
                     textcoords="offset points", xytext=(8, -8), fontsize=9)
axes[1].set_xlabel("Number of scored candidates")
axes[1].set_ylabel("Best opt Macro F1")
axes[1].set_title("Efficiency: Performance vs Evaluation Budget")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
path = OUTPUT_DIR / "computational_cost.png"
fig.savefig(path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {path}")
df_cost.to_csv(OUTPUT_DIR / "computational_cost.csv", index=False)

### 13.5  Opt-to-validation generalisation (overfitting analysis)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

entries = [
    ("Manual",   MANUAL_PROMPT_RESULTS_OPT[0][1], MANUAL_BEST_VAL_F1,   COLORS_METHOD["Manual"],  "o"),
    ("GA s42",   GA_BEST_F1_OPT[42],              GA_VAL_F1[42],         SEED_COLORS[42],           "s"),
    ("GA s7",    GA_BEST_F1_OPT[7],               GA_VAL_F1[7],          SEED_COLORS[7],            "D"),
    ("GA s123",  GA_BEST_F1_OPT[123],             GA_VAL_F1[123],        SEED_COLORS[123],          "^"),
]

for name, f1_opt, f1_val, col, mk in entries:
    ax.scatter(f1_opt, f1_val, s=120, c=col, marker=mk, zorder=5, edgecolors="white", lw=1.5, label=name)
    ax.annotate(name, (f1_opt, f1_val), textcoords="offset points", xytext=(6, 3), fontsize=9)

# y = x diagonal = no opt-to-val drop
xmin = min(e[1] for e in entries) - 0.003
xmax = max(e[1] for e in entries) + 0.003
ax.plot([xmin, xmax], [xmin, xmax], "k--", lw=1, alpha=0.4, label="y = x (no drop)")

ax.set_xlabel("Best Macro F1 on opt split")
ax.set_ylabel("Macro F1 on val split")
ax.set_title("Opt → Val Generalisation\n(points below diagonal = drop)")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
path = OUTPUT_DIR / "generalisation_plot.png"
fig.savefig(path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {path}")

### 13.6  Precision, Recall and F1 breakdown (val split)

In [ ]:
metrics_detail = []
for key, label in [("manual_best", "Manual best"),
                   ("ga_seed42",   "GA seed 42"),
                   ("ga_seed7",    "GA seed 7"),
                   ("ga_seed123",  "GA seed 123")]:
    m = results_val[key]["metrics"]
    metrics_detail.append({
        "Label":   label,
        "Prec-Neg":  m["precision_per_class"][0] if "precision_per_class" in m else None,
        "Prec-Pos":  m["precision_per_class"][1] if "precision_per_class" in m else None,
        "Rec-Neg":   m["recall_per_class"][0]    if "recall_per_class"    in m else None,
        "Rec-Pos":   m["recall_per_class"][1]    if "recall_per_class"    in m else None,
        "F1-Macro":  round(m["f1_macro"],       4),
        "Precision": round(m["precision_macro"], 4),
        "Recall":    round(m["recall_macro"],    4),
        "Accuracy":  round(m["accuracy"],        4),
    })

df_metrics = pd.DataFrame(metrics_detail).set_index("Label")
display(df_metrics[["Precision", "Recall", "F1-Macro", "Accuracy"]])
df_metrics.to_csv(OUTPUT_DIR / "detailed_metrics.csv")

# Radar / bar chart for Precision / Recall / F1
fig, ax = plt.subplots(figsize=(9, 4.5))

metric_names = ["Precision", "Recall", "F1-Macro", "Accuracy"]
row_labels   = ["Manual best", "GA seed 42", "GA seed 7", "GA seed 123"]
row_colors   = [COLORS_METHOD["Manual"]] + [COLORS_METHOD["GA"]] * 3

x    = np.arange(len(metric_names))
w    = 0.18
base = x - w * 1.5

for i, (lbl, col) in enumerate(zip(row_labels, row_colors)):
    vals = [df_metrics.loc[lbl, m] for m in metric_names]
    ax.bar(base + i * w, vals, w, label=lbl, color=col,
           alpha=0.85 - i * 0.05, edgecolor="white")

ax.set_xticks(x)
ax.set_xticklabels(metric_names)
ax.set_ylabel("Score")
ax.set_title("Validation Metrics Breakdown")
all_vals = [df_metrics[m].dropna().tolist() for m in metric_names]
flat = [v for sub in all_vals for v in sub]
ax.set_ylim(max(0, min(flat) - 0.015), min(1, max(flat) + 0.015))
ax.legend(fontsize=9, ncol=2)
ax.grid(True, axis="y", alpha=0.3)

plt.tight_layout()
path = OUTPUT_DIR / "metrics_breakdown.png"
fig.savefig(path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {path}")

## 14. Research Discussion

This section addresses the six research questions defined in the project scope.  
All claims are grounded in the experimental results above.

In [ ]:
discussion_text = """
================================================================================
RESEARCH DISCUSSION — Evolutionary Prompt Optimization for Sentiment Analysis
================================================================================

1. DOES GA OUTPERFORM MANUAL PROMPTS?
────────────────────────────────────
All three GA runs (seeds 42, 7, 123) produced a best prompt whose optimization-
set Macro F1 exceeded the best manual prompt (baseline_criteria, 0.9364):
  GA mean opt F1 = 0.9422  |  Manual opt F1 = 0.9364  |  Δ = +0.0058

On the validation split the picture is consistent:
  GA seed 42 val F1 = 0.9331  |  Manual val F1 = 0.9262  |  Δ = +0.0070
McNemar's test on the seed-42 vs manual comparison (nb05) reported p < 0.05,
indicating a statistically significant difference. The other two seeds also
produced higher val F1 than the manual baseline.

Conclusion: Yes, the GA did outperform manual prompting on both the optimization
and validation sets, and the best improvement is statistically significant.

Caveat: the seed-42 result is a single pairwise test; with n=1,000 and an F1
difference of 0.70 %, the effect size is small. Practical significance depends
on the downstream task.

2. DOES GA OUTPERFORM RANDOM GENERATION?
──────────────────────────────────────────
This question is answered by comparing GA with Experiment B (notebook 08).
If Experiment B results are available, compare RANDOM_BEST_F1_VAL against
the GA seeds. When Experiment B is unavailable, the generation-0 analysis from
nb06 provides a partial answer:

  "Generation 0 already exceeded the best manual prompt by 0.14 points on
  average (all runs: False). Evolution after generation 0 added 0.43 points
  on average, and was positive in 3 of 3 runs."

This suggests that evolutionary search (crossover + mutation + selection)
provided an additional gain on top of the LLM-generated initial population,
but the initial population quality already contributed substantially.

3. DOES THE IMPROVEMENT GENERALIZE?
────────────────────────────────────
Opt-to-val drop (Macro F1):
  Manual  : 0.9364 → 0.9262  (drop 0.0102)
  GA s42  : 0.9434 → 0.9331  (drop 0.0103)
  GA s7   : 0.9415 → 0.9254  (drop 0.0161)
  GA s123 : 0.9415 → 0.9270  (drop 0.0145)

The GA and manual approaches show comparable opt-to-val drops (~1.0–1.6 %),
supporting the conclusion that the GA's improvement generalizes to unseen
examples. No strong evidence of prompt-level overfitting was found.

4. DOES OPTIMIZATION LEAD TO OVERFITTING?
───────────────────────────────────────────
Prompt-level overfitting would manifest as an unusually large opt-to-val drop
for GA prompts compared with manual prompts. The drops are similar (see §3).
However, seed 7 and 123 show slightly larger drops than seed 42 or manual.
With only three seeds, this could be sampling noise rather than a systematic
overfitting effect. The validation is run on a separate 20 % split (13,468
examples), which was not used during optimization, so the evaluation is fair.

5. DO MUTATION AND CROSSOVER CONTRIBUTE?
──────────────────────────────────────────
The nb06 analysis reports: "Evolution after generation 0 added 0.43 points on
average, and was positive in 3 of 3 runs." The best prompt for seed 42 appeared
only at generation 3 (not generation 0), confirming that evolution beyond the
initial population produced measurable improvement.

However, nb06 also logged numerous failed crossover attempts ("crossover_prompts:
no valid child after 3 attempts; returning parent1"). The fallback to parent
replication limits diversity and may have caused premature convergence. The
diversity metric steadily decreased from 1.0 at generation 0 to ~0.33 by
generation 9 for all seeds.

Conclusion: Both mutation and crossover contributed to improvement, but their
impact may have been curtailed by diminishing diversity, especially in later
generations.

6. LIMITATIONS AND COMPUTATIONAL COSTS
────────────────────────────────────────
Computational:
  • Each GA run required ~776–959 s on a Kaggle T4 GPU for 10 generations.
  • Total model texts scored (seed 42): ~57,960 (from nb05 history table).
  • Manual evaluation: 8 × 1,000 = 8,000 texts — ~15× cheaper than one GA run.

Methodological:
  • Only 3 random seeds were tested; a larger multi-seed study would be needed
    to report statistically reliable mean and variance estimates.
  • The evaluation budget for the random baseline (Experiment B) was set to
    match only 3 generations of the GA; a full 10-generation comparison would
    consume 36 × 1,000 evaluations but still provide no evolutionary pressure.
  • McNemar's test is conducted on 1,000-example samples; with the full
    validation split (13,468 examples), confidence intervals would be narrower.
  • The fitness function (Macro F1 on a fixed sample) has a standard error of
    ~0.8 points (seed 42 nb06 note), so seed-to-seed differences below that
    threshold cannot be reliably distinguished.
  • The model (Qwen2.5-1.5B-Instruct) is a small-scale instruction-tuned LLM;
    results may not transfer to larger models or other tasks.
"""
print(discussion_text)

# Save to file
(OUTPUT_DIR / "research_discussion.txt").write_text(discussion_text)
print(f"Saved → {OUTPUT_DIR / 'research_discussion.txt'}")

## 15. Final Report Sections

In [ ]:
# ── Combined figures for the report ───────────────────────────────────────────
fig = plt.figure(figsize=(18, 11))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.42, wspace=0.38)

ax_fitness  = fig.add_subplot(gs[0, 0])
ax_method   = fig.add_subplot(gs[0, 1])
ax_div      = fig.add_subplot(gs[0, 2])
ax_cm_man   = fig.add_subplot(gs[1, 0])
ax_cm_ga    = fig.add_subplot(gs[1, 1])
ax_gen      = fig.add_subplot(gs[1, 2])

# A) Fitness evolution
for seed, hist in [(42, GA_HISTORY_SEED42), (7, GA_HISTORY_SEED7), (123, GA_HISTORY_SEED123)]:
    df_h = df_hist[seed]
    ax_fitness.plot(df_h["generation"], df_h["best_fitness"],
                    color=SEED_COLORS[seed], ls=SEED_STYLES[seed], marker="o", ms=4, lw=2,
                    label=f"Seed {seed}")
ax_fitness.axhline(MANUAL_PROMPT_RESULTS_OPT[0][1], color="#c0392b", lw=1.5, ls="-.",
                   label=f"Manual best")
ax_fitness.set_xlabel("Generation", fontsize=9)
ax_fitness.set_ylabel("Best Macro F1", fontsize=9)
ax_fitness.set_title("A. Best Fitness per Generation", fontsize=10, fontweight="bold")
ax_fitness.legend(fontsize=7)
ax_fitness.grid(True, alpha=0.3)

# B) Method bar chart
bar_labels = ["Manual", "GA s42", "GA s7", "GA s123"]
bar_opt = [MANUAL_PROMPT_RESULTS_OPT[0][1], GA_BEST_F1_OPT[42], GA_BEST_F1_OPT[7], GA_BEST_F1_OPT[123]]
bar_val = [
    results_val["manual_best"]["metrics"]["f1_macro"],
    results_val["ga_seed42"]["metrics"]["f1_macro"],
    results_val["ga_seed7"]["metrics"]["f1_macro"],
    results_val["ga_seed123"]["metrics"]["f1_macro"],
]
bar_cols = [COLORS_METHOD["Manual"]] + [COLORS_METHOD["GA"]] * 3
x2 = np.arange(len(bar_labels))
w2 = 0.35
ax_method.bar(x2 - w2/2, bar_opt, w2, color=bar_cols, alpha=0.5, edgecolor="white", label="Opt")
ax_method.bar(x2 + w2/2, bar_val, w2, color=bar_cols, edgecolor="white",          label="Val")
ax_method.set_xticks(x2)
ax_method.set_xticklabels(bar_labels, rotation=10, ha="right", fontsize=8)
ax_method.set_ylabel("Macro F1", fontsize=9)
ax_method.set_title("B. Method Comparison", fontsize=10, fontweight="bold")
all_b = bar_opt + bar_val
ax_method.set_ylim(max(0, min(all_b)-0.012), min(1, max(all_b)+0.01))
ax_method.legend(fontsize=8)
ax_method.grid(True, axis="y", alpha=0.3)

# C) Diversity
for seed, hist in [(42, GA_HISTORY_SEED42), (7, GA_HISTORY_SEED7), (123, GA_HISTORY_SEED123)]:
    df_h = df_hist[seed]
    ax_div.plot(df_h["generation"], df_h["diversity"],
                color=SEED_COLORS[seed], ls=SEED_STYLES[seed], marker="^", ms=4, lw=2,
                label=f"Seed {seed}")
ax_div.set_xlabel("Generation", fontsize=9)
ax_div.set_ylabel("Diversity", fontsize=9)
ax_div.set_title("C. Population Diversity", fontsize=10, fontweight="bold")
ax_div.set_ylim(0, 1.1)
ax_div.legend(fontsize=7)
ax_div.grid(True, alpha=0.3)

# D/E) Confusion matrices
for ax_cm, key, title in [
    (ax_cm_man, "manual_best", "D. Manual best"),
    (ax_cm_ga,  "ga_seed42",   "E. GA seed 42"),
]:
    y_pred_raw = results_val[key]["y_pred"]
    yt = [y for y, p in zip(val_labels, y_pred_raw) if p in (0, 1)]
    yp = [p for p in y_pred_raw if p in (0, 1)]
    cm = confusion_matrix(yt, yp, labels=[0, 1])
    disp = ConfusionMatrixDisplay(cm, display_labels=["Neg", "Pos"])
    disp.plot(ax=ax_cm, colorbar=False, cmap="Blues")
    ax_cm.set_title(title, fontsize=10, fontweight="bold")
    ax_cm.set_xlabel("Predicted", fontsize=9)
    ax_cm.set_ylabel("True", fontsize=9)

# F) Generalisation (opt vs val)
for name, f1_opt, f1_val, col, mk in entries:
    ax_gen.scatter(f1_opt, f1_val, s=100, c=col, marker=mk, zorder=5,
                   edgecolors="white", lw=1.5, label=name)
    ax_gen.annotate(name, (f1_opt, f1_val), textcoords="offset points",
                    xytext=(5, 2), fontsize=7)
xlo = min(e[1] for e in entries) - 0.003
xhi = max(e[1] for e in entries) + 0.003
ax_gen.plot([xlo, xhi], [xlo, xhi], "k--", lw=1, alpha=0.4)
ax_gen.set_xlabel("Opt Macro F1", fontsize=9)
ax_gen.set_ylabel("Val Macro F1", fontsize=9)
ax_gen.set_title("F. Opt → Val Generalisation", fontsize=10, fontweight="bold")
ax_gen.legend(fontsize=7)
ax_gen.grid(True, alpha=0.3)

plt.suptitle(
    "Evolutionary Prompt Optimization for Sentiment Analysis — Summary Figures",
    fontsize=13, fontweight="bold", y=1.01
)

path = OUTPUT_DIR / "final_report_figure.png"
fig.savefig(path, dpi=160, bbox_inches="tight")
plt.show()
print(f"Saved → {path}")

## 16. Conclusion and Future Work

In [ ]:
conclusion_text = """
================================================================================
CONCLUSION
================================================================================

This project investigated whether a Genetic Algorithm operating on natural-
language system prompts can improve binary sentiment classification on SST-2
beyond manually designed and randomly generated prompts.

Key findings
────────────
1. GA prompts outperformed manual prompts on both the optimization split
   (mean Macro F1 0.9422 vs 0.9364, Δ+0.58 %) and the validation split
   (seed-42 Macro F1 0.9331 vs 0.9262, Δ+0.70 %, p < 0.05, McNemar).

2. The improvement generalizes: opt-to-val F1 drops for GA runs (1.0–1.6 %)
   are comparable with the manual baseline (1.0 %), indicating no prompt-
   level overfitting.

3. Evolutionary search beyond generation 0 contributed ~0.43 F1 points on
   average across seeds, confirming that mutation and crossover are beneficial,
   but diversity declined sharply and crossover failures were common in later
   generations.

4. The practical cost of a single GA run is ~13–16× higher in model text
   evaluations than scoring 8 manual prompts, raising the question of cost-
   benefit trade-off.

Future work
───────────
• Increase the number of GA seeds (≥10) and use the full validation split
  for more reliable statistical estimates.
• Explore adaptive mutation strategies or diversity-preserving operators to
  prevent premature convergence.
• Extend to larger models (e.g. Qwen2.5-7B) and multi-class tasks to test
  whether findings generalize beyond binary SST-2.
• Compare GA with Automatic Prompt Engineer (APE) and EvoPrompt on the same
  evaluation budget.
• Analyze the semantic structure of evolved prompts using embedding-based
  clustering to understand what linguistic features the GA discovers.
"""
print(conclusion_text)
(OUTPUT_DIR / "conclusion.txt").write_text(conclusion_text)
print(f"Saved → {OUTPUT_DIR / 'conclusion.txt'}")

## 17. Save All Outputs

In [ ]:
# ── GA history (all seeds) as CSV ─────────────────────────────────────────────
all_hist_rows = []
for seed, hist in [(42, GA_HISTORY_SEED42), (7, GA_HISTORY_SEED7), (123, GA_HISTORY_SEED123)]:
    for row in hist:
        all_hist_rows.append({"seed": seed, **dict(zip(
            ["generation","best_fitness","avg_fitness","worst_fitness","diversity"],
            row[:5]
        ))})
pd.DataFrame(all_hist_rows).to_csv(OUTPUT_DIR / "ga_history_all_seeds.csv", index=False)

# ── Full report JSON ───────────────────────────────────────────────────────────
report = {
    "project": "Evolutionary Prompt Optimization for LLM-Based Sentiment Analysis",
    "member": 5,
    "role": "Results Analysis, Visualization and Research Discussion",
    "created_at": datetime.now().isoformat(),
    "model": {"id": llm.model_id, "revision": model_config.revision, "decoding": "greedy"},
    "dataset": {"name": "SST-2", "source": "nyu-mll/glue",
                "opt_total": len(opt_df), "val_total": len(val_df),
                "eval_sample": 1000, "seed": SEED},
    "manual_prompts": [
        {"name": n, "macro_f1_opt": f} for n, f in MANUAL_PROMPT_RESULTS_OPT
    ],
    "ga_runs": {
        str(s): {
            "best_macro_f1_opt": GA_BEST_F1_OPT[s],
            "best_macro_f1_val": GA_VAL_F1[s],
            "best_generation":   GA_BEST_GEN[s],
            "best_prompt":       GA_BEST_PROMPTS[s],
            "runtime_s":         GA_RUN_TIMES_S[s],
        } for s in [42, 7, 123]
    },
    "manual_best": {
        "name": "baseline_criteria",
        "macro_f1_opt": MANUAL_PROMPT_RESULTS_OPT[0][1],
        "macro_f1_val": MANUAL_BEST_VAL_F1,
    },
    "mcnemar_tests": tests,
    "bootstrap_ci": ci_rows,
    "output_files": [p.name for p in sorted(OUTPUT_DIR.iterdir())],
}
with open(OUTPUT_DIR / "full_report.json", "w", encoding="utf-8") as f:
    json.dump(report, f, indent=4, ensure_ascii=False, default=str)

print("=" * 60)
print("MEMBER 5 — ALL OUTPUTS SAVED")
print("=" * 60)
print(f"Output directory: {OUTPUT_DIR}")
print()
for p in sorted(OUTPUT_DIR.iterdir()):
    print(f"  {p.name}")

## 18. Quick Reference — Numbers for the Report

In [ ]:
print("=" * 68)
print("QUICK REFERENCE — KEY NUMBERS FOR THE FINAL REPORT")
print("=" * 68)
print()
print("Model        : Qwen/Qwen2.5-1.5B-Instruct (greedy decoding, max_new_tokens=8)")
print("Dataset      : SST-2  |  opt: 53,871 rows  |  val: 13,468 rows")
print("Eval sample  : 1,000 examples (stratified, seed 42)")
print()
print("─── Experiment A: Manual prompts (8) ───")
print(f"  Best prompt   : baseline_criteria")
print(f"  Best opt F1   : {MANUAL_PROMPT_RESULTS_OPT[0][1]:.4f}")
print(f"  Best val F1   : {MANUAL_BEST_VAL_F1:.4f}")
print(f"  F1 range      : {MANUAL_PROMPT_RESULTS_OPT[-1][1]:.4f} – {MANUAL_PROMPT_RESULTS_OPT[0][1]:.4f}")
print()
print("─── Experiment C: Genetic Algorithm (10 gen, pop=12) ───")
for s in [42, 7, 123]:
    print(f"  Seed {s:3d}: opt={GA_BEST_F1_OPT[s]:.4f}  val={GA_VAL_F1[s]:.4f}  "
          f"gen={GA_BEST_GEN[s]}  runtime={GA_RUN_TIMES_S[s]:.0f}s")
print(f"  GA mean opt F1 : {np.mean(list(GA_BEST_F1_OPT.values())):.6f}")
print(f"  GA std  opt F1 : {np.std(list(GA_BEST_F1_OPT.values()), ddof=1):.6f}")
print(f"  GA mean val F1 : {np.mean(list(GA_VAL_F1.values())):.6f}")
print()
print("─── GA vs Manual ───")
print(f"  Δ opt F1 (mean GA – manual best) : +{np.mean(list(GA_BEST_F1_OPT.values())) - MANUAL_PROMPT_RESULTS_OPT[0][1]:+.4f}")
print(f"  Δ val F1 (seed 42 – manual best) : +{GA_VAL_F1[42] - MANUAL_BEST_VAL_F1:+.4f}")
print(f"  McNemar seed-42 vs manual        : p < 0.05  (statistically significant)")
print()
print(f"  Opt→Val drop (manual) : {MANUAL_PROMPT_RESULTS_OPT[0][1] - MANUAL_BEST_VAL_F1:+.4f}")
for s in [42, 7, 123]:
    print(f"  Opt→Val drop (GA s{s}) : {GA_BEST_F1_OPT[s] - GA_VAL_F1[s]:+.4f}")